# 三味線譜変換ツール

五線譜（PDF）→ MusicXML（Audiveris）→ 三味線中間YAML の変換を行います。

**実行順序:**
1. **セル0**: セットアップ（初回のみ）
2. **セル1-a**: PDFアップロード
3. **セル1-a2**: 白塗り選択（任意）→ ✅完了ボタン
4. **セル1-b**: Audiveris変換 → 楽譜表示 → 音声確認
5. **セル2**: 調弦選択
6. **セル3**: 変換 → YAMLダウンロード
7. **セル3-b**: 音域外が残った場合のみ実行（任意）

In [ ]:
# ============================================================
# セル0: セットアップ（最初に一度だけ実行 / 再実行で最新版に更新）
# ============================================================

import subprocess, sys, os
from google.colab import userdata

REPO_NAME = "kamex120/shamisen-converter"
REPO_DIR  = "shamisen-converter"

def run(cmd, **kwargs):
    r = subprocess.run(cmd, capture_output=True, text=True, **kwargs)
    if r.stdout: print(r.stdout.strip())
    if r.returncode != 0:
        print("❌ エラー:", r.stderr.strip())
        raise RuntimeError(f"コマンド失敗: {' '.join(cmd)}")
    return r

# リポジトリのクローン or 更新
token = userdata.get("GITHUB_TOKEN")
repo_url = f"https://{token}@github.com/{REPO_NAME}.git"
if not os.path.exists(REPO_DIR):
    print("📥 リポジトリをクローン中...")
    run(["git", "clone", repo_url, REPO_DIR])
else:
    print("🔄 最新版に更新中...")
    run(["git", "pull"], cwd=REPO_DIR)

os.chdir(REPO_DIR)
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())
print(f"📁 作業ディレクトリ: {os.getcwd()}")

# 依存ライブラリのインストール
print("\n📦 ライブラリをインストール中...")
run([sys.executable, "-m", "pip", "install", "music21", "pyyaml", "pymupdf", "verovio", "-q"])

# --- Audiveris セットアップ ---
DEB_URL   = "https://github.com/Audiveris/audiveris/releases/download/5.10.2/Audiveris-5.10.2-ubuntu22.04-x86_64.deb"
DEB_PATH  = "/tmp/audiveris.deb"
KNOWN_BIN = "/opt/audiveris/bin/Audiveris"

def _find_audiveris():
    r = subprocess.run(["which", "Audiveris"], capture_output=True, text=True)
    if r.returncode == 0 and r.stdout.strip():
        return r.stdout.strip()
    if os.path.isfile(KNOWN_BIN):
        return KNOWN_BIN
    return None

AUDIVERIS_BIN = _find_audiveris()
if AUDIVERIS_BIN:
    print(f"✅ Audiveris インストール済み: {AUDIVERIS_BIN}")
else:
    print("🔧 xvfb をインストール中...")
    subprocess.run(["apt-get", "install", "-y", "xvfb", "-q"], capture_output=True)
    print("📥 Audiveris をダウンロード中...")
    subprocess.run(["wget", "-q", "--show-progress", DEB_URL, "-O", DEB_PATH], check=True)
    print("📦 インストール中...")
    subprocess.run(["dpkg", "-i", DEB_PATH])
    subprocess.run(["apt-get", "install", "-f", "-y", "-q"], capture_output=True)
    os.remove(DEB_PATH)
    AUDIVERIS_BIN = _find_audiveris()
    if AUDIVERIS_BIN:
        print(f"✅ Audiveris セットアップ完了: {AUDIVERIS_BIN}")
    else:
        print("❌ Audiveris バイナリが見つかりません。")

print("\n✅ セットアップ完了")

---
## Step 1: PDF → MusicXML

In [ ]:
# ============================================================
# セル1-a: PDFをアップロード
# ============================================================

from google.colab import files

print("PDFファイルを選択してください")
uploaded = files.upload()

pdf_path = list(uploaded.keys())[0]
selected_rects = []  # 白塗り範囲（セル1-a2で設定、スキップ時は空のまま）
print(f"\n📄 アップロード: {pdf_path}")

In [ ]:
# ============================================================
# セル1-a2: 白塗りツール（任意）
# コード名・歌詞・小節番号を消してAudiverisの誤認識を減らす
# スキップしたい場合はそのまま次のセル1-bへ
# ============================================================

import fitz, base64, json
from IPython.display import HTML, display
from google.colab import output

def _receive_rects(rects_json):
    global selected_rects
    selected_rects = json.loads(rects_json)
    print(f"✅ {len(selected_rects)} 箇所を受信。セル1-bを実行してください")

output.register_callback("store_pdf_rects", _receive_rects)

SCALE = 1.5
doc = fitz.open(pdf_path)
pix = doc[0].get_pixmap(matrix=fitz.Matrix(SCALE, SCALE))
b64 = base64.b64encode(pix.tobytes("png")).decode()

html = f"""
<div style="font-family:sans-serif; margin-bottom:8px;">
  <b>ドラッグして白塗り範囲を選択（複数可）</b><br>
  <span id="status" style="color:#555;">0 箇所選択中</span>
</div>
<div style="margin-bottom:8px;">
  <button onclick="undoLast()" style="margin-right:6px;">↩ 最後を取消</button>
  <button onclick="clearAll()" style="margin-right:6px;">✕ 全クリア</button>
  <button onclick="done()" style="background:#4CAF50;color:white;padding:4px 14px;font-weight:bold;">✅ 完了 → セル1-bへ</button>
</div>
<canvas id="pc" style="cursor:crosshair; border:1px solid #aaa; display:block;"></canvas>
<script>
const SCALE = {SCALE};
const canvas = document.getElementById('pc');
const ctx = canvas.getContext('2d');
let img = new Image(), rects = [], drawing = false, sx = 0, sy = 0;
img.onload = () => {{ canvas.width = img.width; canvas.height = img.height; redraw(); }};
img.src = 'data:image/png;base64,{b64}';
function redraw() {{
  ctx.clearRect(0,0,canvas.width,canvas.height); ctx.drawImage(img,0,0);
  rects.forEach((r,i) => {{
    ctx.fillStyle='rgba(255,255,255,0.55)'; ctx.fillRect(r.x,r.y,r.w,r.h);
    ctx.strokeStyle='red'; ctx.lineWidth=2; ctx.strokeRect(r.x,r.y,r.w,r.h);
    ctx.fillStyle='red'; ctx.font='bold 13px sans-serif'; ctx.fillText(i+1,r.x+3,r.y+14);
  }});
}}
canvas.addEventListener('mousedown', e => {{ const b=canvas.getBoundingClientRect(); sx=e.clientX-b.left; sy=e.clientY-b.top; drawing=true; }});
canvas.addEventListener('mousemove', e => {{
  if(!drawing) return; const b=canvas.getBoundingClientRect(); redraw();
  ctx.strokeStyle='rgba(255,0,0,0.8)'; ctx.lineWidth=2;
  ctx.strokeRect(sx,sy,e.clientX-b.left-sx,e.clientY-b.top-sy);
}});
canvas.addEventListener('mouseup', e => {{
  if(!drawing) return; drawing=false; const b=canvas.getBoundingClientRect();
  const ex=e.clientX-b.left, ey=e.clientY-b.top;
  const x0=Math.min(sx,ex), y0=Math.min(sy,ey), w=Math.abs(ex-sx), h=Math.abs(ey-sy);
  if(w>3&&h>3) rects.push({{x:x0,y:y0,w,h}});
  document.getElementById('status').textContent=rects.length+' 箇所選択中'; redraw();
}});
function undoLast() {{ rects.pop(); redraw(); document.getElementById('status').textContent=rects.length+' 箇所選択中'; }}
function clearAll()  {{ rects=[]; redraw(); document.getElementById('status').textContent='0 箇所選択中'; }}
function done() {{
  const pdf_rects=rects.map(r=>[r.x/SCALE,r.y/SCALE,(r.x+r.w)/SCALE,(r.y+r.h)/SCALE]);
  google.colab.kernel.invokeFunction('store_pdf_rects',[JSON.stringify(pdf_rects)],{{}});
  document.getElementById('status').textContent='✅ 完了（'+rects.length+' 箇所）— セル1-bを実行してください';
}}
</script>
"""
display(HTML(html))

In [ ]:
# ============================================================
# セル1-b: 白塗り適用 → Audiveris → 楽譜表示 → 音声確認
# ============================================================

import os, subprocess, glob
from IPython.display import SVG, Audio, display
from music21 import converter, stream
import verovio

# --- 白塗り適用（セル1-a2で選択した場合のみ）---
if selected_rects:
    import fitz
    doc = fitz.open(pdf_path)
    for page in doc:
        for r in selected_rects:
            page.add_redact_annot(fitz.Rect(*r), fill=(1, 1, 1))
        page.apply_redactions()
    clean_pdf_path = pdf_path.replace(".pdf", "_clean.pdf")
    doc.save(clean_pdf_path)
    pdf_path = clean_pdf_path
    print(f"✅ 白塗り適用: {len(selected_rects)} 箇所 → {clean_pdf_path}")

# --- Audiveris ---
if not AUDIVERIS_BIN:
    raise RuntimeError("Audiveris が見つかりません。セル0を先に実行してください。")

OMR_DIR = "_omr_output/audiveris"
os.makedirs(OMR_DIR, exist_ok=True)
print("🎵 Audiveris で楽譜認識中...")

proc = subprocess.run(
    ["xvfb-run", AUDIVERIS_BIN, "-batch", "-export", "-output", OMR_DIR, "--", pdf_path],
    capture_output=True, text=True
)
if proc.returncode != 0:
    print("❌ Audiveris エラー:", proc.stderr[-500:])
else:
    print(proc.stdout[-300:] or "（出力なし）")

found_mxl = glob.glob(f"{OMR_DIR}/**/*.mxl", recursive=True)
found_xml  = glob.glob(f"{OMR_DIR}/**/*.xml",  recursive=True)
xml_paths  = found_mxl + found_xml

if not xml_paths:
    raise RuntimeError(f"MusicXML が生成されませんでした。出力: {os.listdir(OMR_DIR)}")

if len(xml_paths) == 1:
    musicxml_path = xml_paths[0]
else:
    print(f"📎 {len(xml_paths)} ページをマージ中...")
    merged = stream.Score()
    for xp in xml_paths:
        for part in converter.parse(xp).parts:
            merged.append(part)
    musicxml_path = "_omr_output/merged.xml"
    merged.write("musicxml", fp=musicxml_path)

print(f"✅ MusicXML生成完了 → {musicxml_path}")

# --- 楽譜表示（verovio）---
tk = verovio.toolkit()
tk.setOptions({"pageWidth": 1400, "scale": 35, "adjustPageHeight": 1})
tk.loadFile(musicxml_path)
print(f"楽譜: {tk.getPageCount()} ページ")
for i in range(tk.getPageCount()):
    display(SVG(tk.renderToSVG(i + 1)))

# --- 音声確認（fluidsynth）---
SOUNDFONT = "/usr/share/sounds/sf2/FluidR3_GM.sf2"
MIDI_PATH = "/tmp/preview.mid"
WAV_PATH  = "/tmp/preview.wav"

if not os.path.exists(SOUNDFONT):
    print("🔧 音声合成エンジンをインストール中（初回のみ）...")
    subprocess.run(["apt-get", "install", "-y", "fluidsynth", "fluid-soundfont-gm", "-q"],
                   capture_output=True)

converter.parse(musicxml_path).write("midi", fp=MIDI_PATH)
res = subprocess.run(
    ["fluidsynth", "-ni", SOUNDFONT, MIDI_PATH, "-F", WAV_PATH, "-r", "44100"],
    capture_output=True, text=True
)
if res.returncode == 0:
    print("▶️  再生して認識精度を確認してください:")
    display(Audio(WAV_PATH))
else:
    print("音声生成失敗:", res.stderr[:200])

---
## Step 2: 調弦を選択

In [ ]:
# ============================================================
# セル2: 調弦選択
# ============================================================

import ipywidgets as widgets
from IPython.display import display

tuning_widget = widgets.ToggleButtons(
    options=[
        ("本調子", "honchoshi"),
        ("二上り", "niagari"),
        ("三下り", "sansagari"),
    ],
    description="調弦:",
    button_style="info",
)
display(tuning_widget)

---
## Step 3: MusicXML → 三味線中間YAML

In [ ]:
# ============================================================
# セル3: 変換 → YAML出力 → ダウンロード
# ============================================================

from shamisen_converter import convert_musicxml, load_mapping, build_midi_to_position, to_intermediate_yaml
from google.colab import files

tuning = tuning_widget.value
print(f"調弦: {tuning_widget.label} ({tuning})\n")

result = convert_musicxml(
    musicxml_path=musicxml_path,
    mapping_path="shamisen_mapping.yaml",
    tuning=tuning,
)

total      = len([n for n in result.notes if n.note_name != "rest"])
out_of_range = len([n for n in result.notes if n.out_of_range])
print(f"音符数: {total}")
print(f"音域外: {out_of_range} 件")
for w in result.warnings:
    print(f"  ⚠️  {w}")

# YAML出力
OUTPUT_PATH = "shamisen_output.yaml"
yaml_str = to_intermediate_yaml(result)
with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    f.write(yaml_str)
print(f"\n✅ 変換完了 → {OUTPUT_PATH}")

# ダウンロード
files.download(OUTPUT_PATH)
print(f"⬇️  ダウンロードしました")

if out_of_range > 0:
    print(f"\n⚠️  音域外が {out_of_range} 件残っています。必要であればセル3-bで処理してください")

In [ ]:
# ============================================================
# セル3-b: 音域外の追加処理（セル3で音域外が残った場合のみ実行）
# ============================================================

import ipywidgets as widgets
from IPython.display import display

mapping  = load_mapping("shamisen_mapping.yaml")
midi_map = build_midi_to_position(mapping, tuning)
oor_notes = [n for n in result.notes if n.out_of_range]

if not oor_notes:
    print("✅ 音域外の音符はありません")
else:
    print(f"⚠️  {len(oor_notes)} 件の音域外音符:")
    for n in oor_notes:
        print(f"  offset={n.offset:.1f}  {n.note_name}  MIDI={n.midi}")

    policy_widget = widgets.RadioButtons(
        options=[
            ("すべてオクターブ上げて再変換", "up"),
            ("すべてオクターブ下げて再変換", "down"),
            ("すべて休符として扱う", "skip"),
            ("すべて未解決のまま残す", "keep"),
        ],
        description="処理方針:",
        layout=widgets.Layout(width="400px"),
    )
    apply_btn = widgets.Button(description="適用してYAML再出力", button_style="success")

    def apply_policy(btn):
        policy = policy_widget.value
        for sn in result.notes:
            if not sn.out_of_range:
                continue
            if policy == "up":
                new_midi = sn.midi + 12
                if new_midi in midi_map:
                    s, p = midi_map[new_midi][0]
                    sn.string, sn.position, sn.midi = s, p, new_midi
                    sn.note_name += "(+8va)"; sn.out_of_range = False
            elif policy == "down":
                new_midi = sn.midi - 12
                if new_midi in midi_map:
                    s, p = midi_map[new_midi][0]
                    sn.string, sn.position, sn.midi = s, p, new_midi
                    sn.note_name += "(-8va)"; sn.out_of_range = False
            elif policy == "skip":
                sn.note_name = "rest"; sn.out_of_range = False
        yaml_str = to_intermediate_yaml(result)
        with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
            f.write(yaml_str)
        files.download(OUTPUT_PATH)
        print("✅ 再出力・ダウンロード完了")

    apply_btn.on_click(apply_policy)
    display(policy_widget, apply_btn)

---
## （オプション）MusicXMLを直接アップロードして変換

PDFを使わずMusicXMLファイルから始める場合はこちら。

In [ ]:
# ============================================================
# オプション: MusicXMLを直接アップロード
# ============================================================

from google.colab import files

print("MusicXMLファイル（.xml / .mxl）を選択してください")
uploaded_xml = files.upload()
musicxml_path = list(uploaded_xml.keys())[0]
print(f"\n📄 アップロード: {musicxml_path}")
print("→ セル2（調弦選択）から続けてください")